# C1 GitHub Handover — Rebuild and test
This notebook is a **clean reproducible setup**, not a record of completed image downloads. The source dataset must still be accessible to the Google account. **One-time download of the images is unavoidable** until a persistent copy is available. All changes are made to your designated writable Drive folder; no original shared images are modified.

In [ ]:
%pip install -q timm==1.0.31 pandas Pillow tqdm google-api-python-client google-auth

In [ ]:
from google.colab import drive, auth, files
from pathlib import Path
import sys, zipfile

drive.mount('/content/drive')
auth.authenticate_user()

# C2: Download C1_GitHub_Handover.zip from GitHub, then upload it here.
# This code extracts reusable scripts to a predictable Colab location.
uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise ValueError('Upload exactly one C1_GitHub_Handover.zip file')
PACKAGE_ROOT = Path('/content/C1_handover')
PACKAGE_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(zip_names[0]) as archive:
    archive.extractall(PACKAGE_ROOT)
assert (PACKAGE_ROOT / 'prepare_from_shared_drive.py').exists()
sys.path.insert(0, str(PACKAGE_ROOT))

# Persistent output folder, distinct from Colab's temporary /content folder.
PROJECT_ROOT = Path('/content/drive/MyDrive/Plant_Disease_HW2_C2')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
IMAGE_ROOT = PROJECT_ROOT / 'prepared_data'
print('Permanent dataset destination:', IMAGE_ROOT)

## Prepare the data once
This step lists original Google Drive folders, downloads images to the specified **persistent Drive folder**, validates file types, removes exact duplicates across splits, and writes `splits/{train,valid,test}.csv` and `config/class_weights.pt`. It may take a long time; rerunning it skips already downloaded readable files. Check Google Drive web UI to confirm the files persisted before resetting Colab.

In [ ]:
from prepare_from_shared_drive import prepare
prepare(PROJECT_ROOT, max_workers=4)

## Smoke test DataLoaders and model initialization
The following cells should run after the preparation cell has finished. C2 should train using `train` and `valid` only; reserve `test` for C3.

In [ ]:
from c1_pipeline import create_dataloaders
loaders = create_dataloaders(PROJECT_ROOT, IMAGE_ROOT, batch_size=32, num_workers=0)
for split in ('train','valid','test'):
    image, label = next(iter(loaders[split]))
    print(split, len(loaders[split].dataset), image.shape, label.min().item(), label.max().item())
print('Class weights:', loaders['class_weights'].shape)

In [ ]:
from models import initialize_model, MODEL_NAMES
import torch
for name in MODEL_NAMES:
    model = initialize_model(name, num_classes=9).eval()
    with torch.no_grad():
        output = model(torch.randn(1,3,224,224))
    assert tuple(output.shape)==(1,9)
    print(name, output.shape)
    del model

## Before handing to C2
1. Confirm `PROJECT_ROOT/splits/` and `PROJECT_ROOT/prepared_data/` appear in **Google Drive on the web**.
2. Share the prepared Drive folder with C2; also ensure C2 can access the original shared dataset.
3. Commit `c1_pipeline.py`, `models.py`, `prepare_from_shared_drive.py`, the notebooks, `README.md`, and small config/manifests to GitHub.
4. **Do not** add image files to GitHub. Use the train and validation loaders for tuning; test only for C3.